# Dim Reduction Tuning — Confronto visivo degli embedding UMAP
Notebook per ispezionare visivamente come cambia l'embedding UMAP al variare di `metric`, `n_neighbors`, `min_dist` — a complemento del confronto numerico via trustworthiness (non comparabile direttamente tra metriche diverse, vedi `docs/experiments/dim_reduction_clustering/tuning_dim_s1.1.md` e la chat che ha preceduto questo notebook). Setup/caricamento dati copiati da `quick_dim_reduction_clustering.ipynb`.

## 1. Setup e Caricamento Dati

In [ ]:
import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import umap

if str(Path.cwd().parent) not in sys.path:
    sys.path.append(str(Path.cwd().parent))

from src.analysis.covariates import regress_out_covariate
from src.analysis.distances import binary_pairwise_distance

%matplotlib inline

In [ ]:
# Caricamento Matrice
matrix_path = Path('../data/derived/lesion_matrix/21-07_s1.1/matrix.npy')
X = np.load(matrix_path)
print(f"Shape della matrice lesionale: {X.shape}")

In [ ]:
# Caricamento Metadata e Calcolo Volume
metadata_path = Path('../data/derived/lesion_matrix/21-07_s1.1/metadata.csv')
metadata = pd.read_csv(metadata_path)
metadata['volume_voxel'] = X.sum(axis=1)

print("Metadata caricati con successo!")
metadata[['subject_id', 'dataset', 'volume_voxel']].head()

## 2. Precalcolo delle distanze binarie
Jaccard/dice via `binary_pairwise_distance` (trucco matriciale, secondi invece di minuti - vedi `docs/methods/dimensionality_reduction.md`), calcolate una sola volta e riusate in tutte le celle sotto.

In [ ]:
X_jaccard_dist = binary_pairwise_distance(X, "jaccard")
X_dice_dist = binary_pairwise_distance(X, "dice")
lesion_load_voxels = X.sum(axis=1)  # stesso covariate usato da regress_out_volume in produzione
print("Matrici di distanza precomputate:", X_jaccard_dist.shape, X_dice_dist.shape)

## 3. Helper: fit UMAP + plot a griglia
Definite una sola volta, richiamate in ogni sezione sotto — così ogni cella successiva produce il proprio plot senza sovrascrivere quelli precedenti.

In [ ]:
def fit_umap(X_input, metric_arg, n_neighbors, min_dist, random_state=0):
    """metric_arg='precomputed' quando X_input è già una matrice di distanza (jaccard/dice)."""
    return umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=2,
        metric=metric_arg,
        random_state=random_state,
    ).fit_transform(X_input)


def plot_embedding_grid(embeddings_and_titles, suptitle, ncols=None):
    """Un subplot per (embedding, titolo) - scatter semplice, nessuna colorazione per metadati."""
    n = len(embeddings_and_titles)
    ncols = ncols or n
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows), squeeze=False)
    axes = axes.flatten()
    for ax, (embedding, title) in zip(axes, embeddings_and_titles):
        ax.scatter(embedding[:, 0], embedding[:, 1], s=12, alpha=0.5)
        ax.set_title(title)
        ax.set_xlabel("UMAP 1")
        ax.set_ylabel("UMAP 2")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(suptitle)
    fig.tight_layout()
    plt.show()

In [ ]:
# Parametri fissi di default (produzione attuale, config/registry/params_reduction.json)
BASE_N_NEIGHBORS = 15
BASE_MIN_DIST = 0.0
RANDOM_STATE = 0

# Griglia usata per i confronti sotto (n_neighbors: stessa del tuning_grid attuale;
# min_dist: non ancora in tuning_grid, valori scelti per coprire lo spettro compatto->sparso)
N_NEIGHBORS_GRID = [5, 15, 30, 50, 100]
MIN_DIST_GRID = [0.0, 0.1, 0.25, 0.5, 0.8]

## 4. Effetto della metrica
`n_neighbors`/`min_dist` fissi ai valori di produzione. 4 pannelli: jaccard, dice, euclidean (senza e con `regress_out_volume`) — l'embedding euclideo è calcolato una sola volta, la versione "regredita" è lo stesso embedding con `regress_out_covariate` applicato dopo, esattamente come fa `dim_reduction.py` in produzione.

In [ ]:
embedding_jaccard = fit_umap(X_jaccard_dist, "precomputed", BASE_N_NEIGHBORS, BASE_MIN_DIST, RANDOM_STATE)
embedding_dice = fit_umap(X_dice_dist, "precomputed", BASE_N_NEIGHBORS, BASE_MIN_DIST, RANDOM_STATE)
embedding_euclidean = fit_umap(X, "euclidean", BASE_N_NEIGHBORS, BASE_MIN_DIST, RANDOM_STATE)
embedding_euclidean_regressed = regress_out_covariate(embedding_euclidean, lesion_load_voxels)

plot_embedding_grid(
    [
        (embedding_jaccard, "jaccard"),
        (embedding_dice, "dice"),
        (embedding_euclidean, "euclidean (regress_out_volume=False)"),
        (embedding_euclidean_regressed, "euclidean (regress_out_volume=True)"),
    ],
    suptitle=f"UMAP per metrica (n_neighbors={BASE_N_NEIGHBORS}, min_dist={BASE_MIN_DIST})",
    ncols=2,
)

## 5. Metrica fissa: euclidean — effetto di `n_neighbors` e `min_dist`

In [ ]:
embeddings_nn_euclidean = [
    (fit_umap(X, "euclidean", n, BASE_MIN_DIST, RANDOM_STATE), f"n_neighbors={n}")
    for n in N_NEIGHBORS_GRID
]
plot_embedding_grid(
    embeddings_nn_euclidean,
    suptitle=f"UMAP euclidean (min_dist={BASE_MIN_DIST}) al variare di n_neighbors",
    ncols=5,
)

In [ ]:
embeddings_md_euclidean = [
    (fit_umap(X, "euclidean", BASE_N_NEIGHBORS, md, RANDOM_STATE), f"min_dist={md}")
    for md in MIN_DIST_GRID
]
plot_embedding_grid(
    embeddings_md_euclidean,
    suptitle=f"UMAP euclidean (n_neighbors={BASE_N_NEIGHBORS}) al variare di min_dist",
    ncols=5,
)

## 5b. Metrica fissa: euclidean + `regress_out_volume=True` — effetto di `n_neighbors` e `min_dist`
Stessa griglia della sezione 5, ma ogni embedding ha `regress_out_covariate` applicato prima del plot (stesso identico step della modalità produzione con `regress_out_volume: true`).

In [ ]:
embeddings_nn_euclidean_regressed = [
    (regress_out_covariate(fit_umap(X, "euclidean", n, BASE_MIN_DIST, RANDOM_STATE), lesion_load_voxels), f"n_neighbors={n}")
    for n in N_NEIGHBORS_GRID
]
plot_embedding_grid(
    embeddings_nn_euclidean_regressed,
    suptitle=f"UMAP euclidean, regress_out_volume=True (min_dist={BASE_MIN_DIST}) al variare di n_neighbors",
    ncols=5,
)

In [ ]:
embeddings_md_euclidean_regressed = [
    (regress_out_covariate(fit_umap(X, "euclidean", BASE_N_NEIGHBORS, md, RANDOM_STATE), lesion_load_voxels), f"min_dist={md}")
    for md in MIN_DIST_GRID
]
plot_embedding_grid(
    embeddings_md_euclidean_regressed,
    suptitle=f"UMAP euclidean, regress_out_volume=True (n_neighbors={BASE_N_NEIGHBORS}) al variare di min_dist",
    ncols=5,
)

## 6. Metrica fissa: jaccard — effetto di `n_neighbors` e `min_dist`

In [ ]:
embeddings_nn_jaccard = [
    (fit_umap(X_jaccard_dist, "precomputed", n, BASE_MIN_DIST, RANDOM_STATE), f"n_neighbors={n}")
    for n in N_NEIGHBORS_GRID
]
plot_embedding_grid(
    embeddings_nn_jaccard,
    suptitle=f"UMAP jaccard (min_dist={BASE_MIN_DIST}) al variare di n_neighbors",
    ncols=5,
)

In [ ]:
embeddings_md_jaccard = [
    (fit_umap(X_jaccard_dist, "precomputed", BASE_N_NEIGHBORS, md, RANDOM_STATE), f"min_dist={md}")
    for md in MIN_DIST_GRID
]
plot_embedding_grid(
    embeddings_md_jaccard,
    suptitle=f"UMAP jaccard (n_neighbors={BASE_N_NEIGHBORS}) al variare di min_dist",
    ncols=5,
)

## 7. Prossimi passi
Da decidere insieme: se fissare `dice` allo stesso modo di `jaccard` sopra, se esplorare altre combinazioni (es. `regress_out_volume` incrociato con `n_neighbors`), o fermarsi qui e scegliere i parametri di produzione guardando questi plot + la tabella `tuning_results.csv`.